In [1]:
import sys
import tempfile
import urllib.request

import librosa

# Add the parent directory to the path so we can import from src
sys.path.insert(0, "..")

from src.melt.processing_melt import MELT_SPECIAL_TOKENS, MELTProcessor
from transformers import AutoFeatureExtractor, AutoTokenizer

/mnt/home/giuseppe/mydata/melt-proj/training/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup: Load Components and Audio Sample

In [2]:
# Audio sample URL for testing
AUDIO_SAMPLE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/guess_age_gender.wav"

# Download and load audio sample
def load_audio_sample(url):
    """Load audio sample from URL using librosa."""
    with urllib.request.urlopen(url) as response:
        audio_bytes = response.read()
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
        tmp_file.write(audio_bytes)
        tmp_path = tmp_file.name
    
    audio, sr = librosa.load(tmp_path, sr=16000)
    return audio

audio_sample = load_audio_sample(AUDIO_SAMPLE_URL)
print(f"Audio sample shape: {audio_sample.shape}")
print(f"Audio duration: {len(audio_sample) / 16000:.2f} seconds")

Audio sample shape: (144000,)
Audio duration: 9.00 seconds


In [3]:
# Load feature extractor and tokenizer
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")

# Create processor
processor = MELTProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Processor created successfully!")
print(f"Audio token: {processor.audio_token}")
print(f"Audio BOS token: {processor.audio_bos_token}")
print(f"Audio EOS token: {processor.audio_eos_token}")

Processor created successfully!
Audio token: <|AUDIO|>
Audio BOS token: <|audio_bos|>
Audio EOS token: <|audio_eos|>


In [4]:
def display_result(result, processor, title="Result"):
    """Helper function to display processor output."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    
    print(f"\nKeys in result: {list(result.keys())}")
    
    # Handle both list and tensor inputs
    input_ids = result["input_ids"]
    if hasattr(input_ids, "shape"):
        print(f"\ninput_ids shape: {input_ids.shape}")
        num_samples = input_ids.shape[0]
    else:
        print(f"\ninput_ids: list of {len(input_ids)} samples")
        num_samples = len(input_ids)
    
    if "input_features" in result:
        features = result["input_features"]
        if hasattr(features, "shape"):
            print(f"input_features shape: {features.shape}")
        else:
            print(f"input_features: list of {len(features)} items")
    
    print(f"\n--- Decoded text for each sample ---")
    for i in range(num_samples):
        if hasattr(input_ids, "shape"):
            ids = input_ids[i]
        else:
            ids = input_ids[i]
        decoded = processor.decode(ids)
        print(f"\nSample {i+1}:")
        print(f"  Token count: {len(ids)}")
        print(f"  Decoded (first 500 chars):")
        print(f"  {decoded[:500]}..." if len(decoded) > 500 else f"  {decoded}")

## 1. Text Only Processing

### 1.1 Simple Text (No Chat Template)

In [5]:
text = "Hello, how are you today?"
result = processor(text=text)

display_result(result, processor, "Simple Text (No Chat Template)")


Simple Text (No Chat Template)

Keys in result: ['input_ids', 'attention_mask']

input_ids: list of 1 samples

--- Decoded text for each sample ---

Sample 1:
  Token count: 7
  Decoded (first 500 chars):
  Hello, how are you today?


### 1.2 Text with Chat Template

In [6]:
messages = [
    {"role": "user", "content": "Hello, how are you?"},
    {"role": "assistant", "content": "I'm doing well, thank you!"},
]

# Apply chat template
text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

result = processor(text=text_with_template)
display_result(result, processor, "Text with Chat Template")

Raw text after applying chat template:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you!<|im_end|>
<|im_start|>assistant



Text with Chat Template

Keys in result: ['input_ids', 'attention_mask']

input_ids: list of 1 samples

--- Decoded text for each sample ---

Sample 1:
  Token count: 38
  Decoded (first 500 chars):
  <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you!<|im_end|>
<|im_start|>assistant



## 2. Single Audio Processing

### 2.1 Single Audio (No Chat Template)

In [7]:
audio_token = processor.audio_token
text = f"Transcribe the following audio: {audio_token}"

print(f"Input text: {text}")
print()

result = processor(text=text, audio=audio_sample)
display_result(result, processor, "Single Audio (No Chat Template)")

Input text: Transcribe the following audio: <|AUDIO|>


Single Audio (No Chat Template)

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids: list of 1 samples
input_features shape: (1, 449, 160)

--- Decoded text for each sample ---

Sample 1:
  Token count: 119
  Decoded (first 500 chars):
  Transcribe the following audio: <|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|>...

Single Audio (No Chat Template)

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids: list of 1 sa

### 2.2 Single Audio with Chat Template

In [8]:
audio_token = processor.audio_token
messages = [
    {"role": "user", "content": f"Transcribe the following audio: {audio_token}"},
]

text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

result = processor(text=text_with_template, audio=audio_sample)
display_result(result, processor, "Single Audio with Chat Template")

Raw text after applying chat template:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Transcribe the following audio: <|AUDIO|><|im_end|>
<|im_start|>assistant



Single Audio with Chat Template

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids: list of 1 samples
input_features shape: (1, 449, 160)

--- Decoded text for each sample ---

Sample 1:
  Token count: 138
  Decoded (first 500 chars):
  <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Transcribe the following audio: <|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDI...

Single Audio w

## 3. Multiple Audios Per Sample

### 3.1 Three Audios (No Chat Template)

In [10]:
audio_token = processor.audio_token
text = f"{audio_token} What is said here? {audio_token} And in this one? {audio_token} Summarize all."
audios = [audio_sample, audio_sample, audio_sample]

print(f"Input text: {text}")
print(f"Number of audios: {len(audios)}")
print()

result = processor(text=text, audio=audios)
display_result(result, processor, "Three Audios (No Chat Template)")

Input text: <|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.
Number of audios: 3


Three Audios (No Chat Template)

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids: list of 1 samples
input_features shape: (3, 449, 160)

--- Decoded text for each sample ---

Sample 1:
  Token count: 353
  Decoded (first 500 chars):
  <|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUD...

Three Audios (No Chat Template)

Keys in result: ['input_ids', 'attention_mask', 'input_fe

### 3.2 Three Audios with Chat Template

In [9]:
audio_token = processor.audio_token
messages = [
    {
        "role": "user",
        "content": f"{audio_token} What is said here? {audio_token} And in this one? {audio_token} Summarize all.",
    },
]

text_with_template = processor.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Raw text after applying chat template:")
print(text_with_template)
print()

audios = [audio_sample, audio_sample, audio_sample]
result = processor(text=text_with_template, audio=audios)
display_result(result, processor, "Three Audios with Chat Template")

Raw text after applying chat template:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.<|im_end|>
<|im_start|>assistant



Three Audios with Chat Template

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids: list of 1 samples
input_features shape: (3, 449, 160)

--- Decoded text for each sample ---

Sample 1:
  Token count: 372
  Decoded (first 500 chars):
  <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|>

### 3.3 Two Audios for Comparison

In [ ]:
audio_token = processor.audio_token
text = f"Compare {audio_token} with {audio_token}"
audios = [audio_sample, audio_sample]

print(f"Input text: {text}")
print(f"Number of audios: {len(audios)}")
print()

result = processor(text=text, audio=audios)
display_result(result, processor, "Two Audios for Comparison")

## 4. Batch Processing with Different Audio Counts

### 4.1 Batch: Sample 1 (3 audios) + Sample 2 (1 audio)

In [12]:
audio_token = processor.audio_token

# Sample 1 has 3 audios
text1 = f"{audio_token} What is said here? {audio_token} And in this one? {audio_token} Summarize all."
# Sample 2 has 1 audio
text2 = f"Transcribe: {audio_token}"

texts = [text1, text2]
# 3 audios for sample 1, 1 audio for sample 2 = 4 total
audios = [audio_sample, audio_sample, audio_sample, audio_sample]

print("Sample 1 text:", text1)
print("Sample 2 text:", text2)
print(f"Total audios: {len(audios)}")
print()

result = processor(text=texts, audio=audios, padding=True)
display_result(result, processor, "Batch: 3 audios + 1 audio")

Sample 1 text: <|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.
Sample 2 text: Transcribe: <|AUDIO|>
Total audios: 4


Batch: 3 audios + 1 audio

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids: list of 2 samples
input_features shape: (4, 449, 160)

--- Decoded text for each sample ---

Sample 1:
  Token count: 353
  Decoded (first 500 chars):
  <|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUD...

Sample 2:
  Token count: 353
  Decoded (first 500 chars):
  

### 4.2 Batch: Sample 1 (1 audio) + Sample 2 (2 audios)

In [ ]:
audio_token = processor.audio_token

# Sample 1 has 1 audio
text1 = f"Single audio sample: {audio_token}"
# Sample 2 has 2 audios
text2 = f"Compare {audio_token} with {audio_token}"

texts = [text1, text2]
# 1 audio for sample 1, 2 audios for sample 2 = 3 total
audios = [audio_sample, audio_sample, audio_sample]

print("Sample 1 text:", text1)
print("Sample 2 text:", text2)
print(f"Total audios: {len(audios)}")
print()

result = processor(text=texts, audio=audios, padding=True)
display_result(result, processor, "Batch: 1 audio + 2 audios")

### 4.3 Batch with Chat Template: Mixed Audio Counts

In [ ]:
audio_token = processor.audio_token

# Sample 1: multi-audio with chat template
messages1 = [
    {
        "role": "user",
        "content": f"{audio_token} Describe this. {audio_token} And this.",
    },
]
text1 = processor.tokenizer.apply_chat_template(
    messages1, tokenize=False, add_generation_prompt=True
)

# Sample 2: single audio with chat template
messages2 = [
    {"role": "user", "content": f"What do you hear? {audio_token}"},
]
text2 = processor.tokenizer.apply_chat_template(
    messages2, tokenize=False, add_generation_prompt=True
)

texts = [text1, text2]
# 2 audios for sample 1, 1 audio for sample 2 = 3 total
audios = [audio_sample, audio_sample, audio_sample]

print("Sample 1 (chat template applied):")
print(text1)
print("\nSample 2 (chat template applied):")
print(text2)
print(f"\nTotal audios: {len(audios)}")
print()

result = processor(text=texts, audio=audios, padding=True)
display_result(result, processor, "Batch with Chat Template: 2 audios + 1 audio")

## 5. Examining Audio Token Expansion

In [ ]:
audio_token = processor.audio_token
text = f"Before {audio_token} after"

print(f"Input text: {text}")
print()

result = processor(text=text, audio=audio_sample)

# Count occurrences of audio token in decoded output
decoded = processor.decode(result["input_ids"][0])
audio_token_count = decoded.count(audio_token)

print(f"Audio token count in decoded output: {audio_token_count}")
print(f"\nFull decoded output:")
print(decoded)

## 6. Summary: Special Tokens Available

In [ ]:
print("MELT Special Tokens:")
print("=" * 40)
for name, token in MELT_SPECIAL_TOKENS.items():
    token_id = processor.tokenizer.convert_tokens_to_ids(token)
    print(f"{name:20s}: {token:15s} (ID: {token_id})")

## 7. Comparison with Qwen2AudioProcessor and Phi4MultimodalProcessor

Let's compare how MELTProcessor handles batched inputs with different audio counts against the official Qwen2AudioProcessor and Phi4MultimodalProcessor implementations.

### 7.1 Qwen2AudioProcessor Comparison

In [10]:
# Load Qwen2AudioProcessor
from transformers import Qwen2AudioProcessor

qwen2_processor = Qwen2AudioProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")
print("Qwen2AudioProcessor loaded!")
print(f"Audio token: {qwen2_processor.audio_token}")
print(f"Audio BOS token: {qwen2_processor.audio_bos_token}")
print(f"Audio EOS token: {qwen2_processor.audio_eos_token}")

Qwen2AudioProcessor loaded!
Audio token: <|AUDIO|>
Audio BOS token: <|audio_bos|>
Audio EOS token: <|audio_eos|>


In [11]:
# Qwen2Audio: Batch with 3 audios in sample 1, 1 audio in sample 2
qwen_audio_token = qwen2_processor.audio_token

# Sample 1: 3 audios
text1_qwen = f"{qwen_audio_token} What is said here? {qwen_audio_token} And in this one? {qwen_audio_token} Summarize all."
# Sample 2: 1 audio  
text2_qwen = f"Transcribe: {qwen_audio_token}"

texts_qwen = [text1_qwen, text2_qwen]
# 3 audios for sample 1, 1 audio for sample 2 = 4 total
audios_qwen = [audio_sample, audio_sample, audio_sample, audio_sample]

print("Qwen2Audio Input:")
print(f"Sample 1: {text1_qwen}")
print(f"Sample 2: {text2_qwen}")
print(f"Total audios: {len(audios_qwen)}")
print()

try:
    result_qwen = qwen2_processor(text=texts_qwen, audio=audios_qwen, padding=True)
    display_result(result_qwen, qwen2_processor, "Qwen2AudioProcessor: Batch 3+1 audios")
except Exception as e:
    print(f"Error with Qwen2AudioProcessor: {e}")

It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


Qwen2Audio Input:
Sample 1: <|AUDIO|> What is said here? <|AUDIO|> And in this one? <|AUDIO|> Summarize all.
Sample 2: Transcribe: <|AUDIO|>
Total audios: 4


Qwen2AudioProcessor: Batch 3+1 audios

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids: list of 2 samples
input_features shape: (4, 128, 3000)

--- Decoded text for each sample ---

Sample 1:
  Token count: 698
  Decoded (first 500 chars):
  <|audio_bos|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><...

Sample 2:
  Token count: 698
  Decoded 

### 7.2 Phi4MultimodalProcessor Comparison

In [ ]:
from transformers import AutoProcessor

# Load Phi4MultimodalProcessor
phi4_processor = AutoProcessor.from_pretrained("microsoft/Phi-4-multimodal-instruct", trust_remote_code=True)
print("Phi4MultimodalProcessor loaded!")
# print(f"Audio token: {phi4_processor.audio_token}")

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-multimodal-instruct:
- processing_phi4mm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/mnt/home/giuseppe/mydata/melt-proj/training/venv/lib/python3.11/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
/mnt/home/giuseppe/mydata/melt-proj/training/venv/lib/python3.11/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a 

Phi4MultimodalProcessor loaded!


AttributeError: 'Phi4MMProcessor' object has no attribute 'audio_token'

In [21]:
# Phi4Multimodal: Batch with 3 audios in sample 1, 1 audio in sample 2
# phi4_audio_token = phi4_processor.audio_token


# Sample 1: 3 audios
text1_phi4 = f"<audio_1> What is said here? <audio_2> And in this one? <audio_3> Summarize all."
# Sample 2: 1 audio
text2_phi4 = f"Transcribe: <audio_1>"

texts_phi4 = [text1_phi4, text2_phi4]
# 3 audios for sample 1, 1 audio for sample 2 = 4 total
audios_phi4 = [[audio_sample, audio_sample, audio_sample], [audio_sample]]

print("Phi4Multimodal Input:")
print(f"Sample 1: {text1_phi4}")
print(f"Sample 2: {text2_phi4}")
print(f"Total audios: {len(audios_phi4)}")
print()

try:
    result_phi4 = phi4_processor(text=texts_phi4, audios=audios_phi4)
    print(result_phi4)
    display_result(result_phi4, phi4_processor, "Phi4MultimodalProcessor: Batch 3+1 audios")
except Exception as e:
    print(f"Error with Phi4MultimodalProcessor: {e}")

Phi4Multimodal Input:
Sample 1: <audio_1> What is said here? <audio_2> And in this one? <audio_3> Summarize all.
Sample 2: Transcribe: <audio_1>
Total audios: 2

Error with Phi4MultimodalProcessor: too many values to unpack (expected 2)


### 7.3 MELTProcessor Comparison (same input)

In [38]:
# MELTProcessor: Same batch with 3 audios in sample 1, 1 audio in sample 2
melt_audio_token = processor.audio_token

# Sample 1: 3 audios
text1_melt = f"{melt_audio_token} What is said here? {melt_audio_token} And in this one? Summarize all."
# text1_melt = f"{melt_audio_token} What is said here? {melt_audio_token} And in this one? {melt_audio_token} Summarize all."
# Sample 2: 1 audio
text2_melt = f"Transcribe: {melt_audio_token}"

texts_melt = [text1_melt, text2_melt]
# 3 audios for sample 1, 1 audio for sample 2 = 4 total
audios_melt = [[audio_sample, audio_sample, audio_sample], [audio_sample]]

print("MELTProcessor Input:")
print(f"Sample 1: {text1_melt}")
print(f"Sample 2: {text2_melt}")
print(f"Total audios: {len(audios_melt)}")
print()

result_melt = processor(text=texts_melt, audio=audios_melt, padding=True, return_tensors="pt")
display_result(result_melt, processor, "MELTProcessor: Batch 3+1 audios")

MELTProcessor Input:
Sample 1: <|AUDIO|> What is said here? <|AUDIO|> And in this one? Summarize all.
Sample 2: Transcribe: <|AUDIO|>
Total audios: 2


MELTProcessor: Batch 3+1 audios

Keys in result: ['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

input_ids shape: torch.Size([2, 240])
input_features shape: torch.Size([2, 449, 160])

--- Decoded text for each sample ---

Sample 1:
  Token count: 240
  Decoded (first 500 chars):
  <|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUDIO|><|AUD...

Sample 2:
  Token count: 240
  D

In [40]:
list(result_melt.keys())

['input_ids', 'attention_mask', 'input_features', 'feature_attention_mask']

In [ ]:
result_melt["feature_attention_mask"].shape

In [ ]:
result_melt["feature_attention_mask"].sum(-1)

In [ ]:
result_melt

In [34]:
result_melt["attention_mask"].shape

torch.Size([2, 242])

In [31]:
print(sum(result_melt["attention_mask"][0]))
print(sum(result_melt["attention_mask"][1]))

242
5


### 7.4 Summary Comparison

Compare the output keys and shapes across all three processors:

In [ ]:
def compare_outputs(results_dict):
    """Compare outputs from different processors."""
    print("=" * 80)
    print("COMPARISON SUMMARY")
    print("=" * 80)
    
    for name, result in results_dict.items():
        if result is None:
            print(f"\n{name}: Not available (failed to load/process)")
            continue
            
        print(f"\n{name}:")
        print(f"  Keys: {list(result.keys())}")
        for key, value in result.items():
            if hasattr(value, "shape"):
                print(f"  {key}: shape = {value.shape}")
            elif isinstance(value, list):
                if len(value) > 0 and hasattr(value[0], "__len__"):
                    print(f"  {key}: list of {len(value)} items, first item length = {len(value[0])}")
                else:
                    print(f"  {key}: list of {len(value)} items")
            else:
                print(f"  {key}: {type(value).__name__}")

# Compare all results
results_to_compare = {
    "MELTProcessor": result_melt,
}

# Add Qwen2Audio if it worked
try:
    results_to_compare["Qwen2AudioProcessor"] = result_qwen
except NameError:
    results_to_compare["Qwen2AudioProcessor"] = None

# Add Phi4Multimodal if it worked
try:
    results_to_compare["Phi4MultimodalProcessor"] = result_phi4
except NameError:
    results_to_compare["Phi4MultimodalProcessor"] = None

compare_outputs(results_to_compare)